## chương trình

### Thư viện

huấn luyện một mô hình ngôn ngữ lớn (LLM) với dữ liệu instruction
để thực hiện các câu hỏi trắc nghiệm trong bài đọc hiểu thuộc kỳ thi SAT

In [1]:
!pip install -q -U bitsandbytes
!pip install -q -U datasets
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U loralib
!pip install -q -U einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔧 5. PEFT – Parameter-Efficient Fine-Tuning
from peft import ...
Thư viện PEFT (từ Hugging Face) hỗ trợ fine-tuning hiệu quả thông qua các kỹ thuật như:

LoRA (Low-Rank Adaptation): chỉ huấn luyện một phần nhỏ tham số → tiết kiệm tài nguyên và tránh overfitting.

LoraConfig: cấu hình số rank, alpha, target layers,...

PeftModel, get_peft_model: bọc mô hình với các tham số LoRA.

prepare_model_for_kbit_training: chuẩn bị mô hình đã được quantize để fine-tune đúng cách.

In [2]:
import json
import os
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers

from pprint import pprint
from tqdm import tqdm
from datasets import load_dataset, Dataset

from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training
)
from transformers import (
    AutoConfig,                # Đọc cấu hình mô hình từ huggingface hub hoặc local
    AutoModelForCausalLM,      # Tải mô hình ngôn ngữ dạng sinh (generative, như GPT)
    AutoTokenizer,             # Tải tokenizer phù hợp với mô hình
    BitsAndBytesConfig         # Cấu hình quantization 8bit / 4bit
)

In [ ]:
from huggingface_hub import notebook_login
# Tự động mở hộp thoại để bạn dán token vào
notebook_login()

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
# MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.gradient_checkpointing_enable()


In [6]:
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "v_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

In [7]:
# model = prepare_model_for_kbit_training(model)
# peft_config = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     target_modules=[
#         "q_proj",
#         "v_proj",
#     ],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# prepare_model_for_kbit_training(model):
# Hàm này chuẩn bị mô hình đã được lượng tử hóa (quantized) bằng bitsandbytes
# để sẵn sàng cho quá trình fine-tuning. Khi fine-tuning mô hình đã được lượng tử hóa
# ở chế độ 4-bit (như được cấu hình trong bnb_config ở phần trước),
# cần thực hiện một số thao tác đặc biệt để gradient được tính toán và cập nhật đúng cách.
# Hàm này thực hiện các bước cần thiết đó, ví dụ như enabling gradient checkpointing
# và xử lý các module lượng tử hóa.

# peft_config = LoraConfig(...):
# Đoạn code này định nghĩa cấu hình cho kỹ thuật LoRA (Low-Rank Adaptation),
# một phương pháp Parameter-Efficient Fine-Tuning (PEFT).
# LoRA thêm các ma trận bậc thấp vào một số lớp của mô hình gốc và chỉ huấn luyện
# các ma trận nhỏ này, giữ nguyên phần lớn tham số của mô hình ban đầu.
# Điều này giúp giảm đáng kể số lượng tham số cần huấn luyện, tiết kiệm tài nguyên
# tính toán và bộ nhớ, đồng thời giảm thiểu rủi ro overfitting.

# Các tham số trong LoraConfig:
# - r=8: Đây là rank của các ma trận bậc thấp được thêm vào. Rank càng cao thì
#   khả năng biểu đạt của các ma trận LoRA càng lớn, nhưng số lượng tham số
#   huấn luyện cũng tăng lên. Giá trị 8 là khá phổ biến và thường cho kết quả tốt.
# - lora_alpha=16: Tham số này liên quan đến việc scaling (nhân tỷ lệ) các trọng
#   số LoRA. Công thức scaling thường là `lora_alpha / r`. Giá trị 16 ở đây kết
#   hợp với r=8 cho tỷ lệ scaling là 16/8 = 2.
# - target_modules=["q_proj", "v_proj"]: Đây là danh sách các module (lớp)
#   trong mô hình gốc mà LoRA sẽ được áp dụng. Trong trường hợp này, LoRA
#   được áp dụng vào các ma trận chiếu query ("q_proj") và value ("v_proj")
#   trong cơ chế self-attention của mô hình. Đây là các vị trí phổ biến và
#   thường hiệu quả khi áp dụng LoRA cho các mô hình Transformer.
# - lora_dropout=0.05: Tỷ lệ dropout được áp dụng cho các ma trận LoRA.
#   Dropout giúp chống overfitting.
# - bias="none": Chỉ định cách xử lý bias weights. "none" nghĩa là bias weights
#   không bị ảnh hưởng bởi LoRA. Các lựa chọn khác có thể là "all" hoặc "lora_only".
# - task_type="CAUSAL_LM": Xác định loại tác vụ mà mô hình được fine-tuning cho.
#   "CAUSAL_LM" (Causal Language Modeling) là tác vụ dự đoán token tiếp theo,
#   phù hợp với các mô hình ngôn ngữ sinh như Llama.

# Sau khi định nghĩa peft_config, cấu hình này sẽ được sử dụng cùng với
# hàm get_peft_model() (như trong dòng code tiếp theo trong file gốc
# mà bạn cung cấp) để bọc mô hình gốc (đã được chuẩn bị bằng
# prepare_model_for_kbit_training) với các lớp LoRA dựa trên cấu hình đã cho.


### Data và xây dựng các hàm

In [ ]:
data = load_dataset("emozilla/sat-reading")

In [9]:
def extract_sections(text):
    """
    Extracts passage, question, choices, and answer letter from the text block.

    Args:
        text (str): The input text block containing the SAT reading comprehension problem.

    Returns:
        dict: A dictionary containing the extracted sections:
              "passage": The reading passage.
              "question": The question text.
              "choices": A list of answer choices (including letters).
              "answer_letter": The correct answer letter (e.g., "A", "B").
    """
    # Khởi tạo dictionary để lưu trữ các phần đã trích xuất
    sections = {
        "passage": "",       # Khởi tạo phần bài đọc là chuỗi rỗng
        "question": "",      # Khởi tạo phần câu hỏi là chuỗi rỗng
        "choices": [],       # Khởi tạo phần các lựa chọn là danh sách rỗng
        "answer_letter": ""  # Khởi tạo phần chữ cái đáp án là chuỗi rỗng
    }

    # Extract the answer letter from the end
    # Tìm kiếm chuỗi "Answer: " theo sau là một chữ cái từ A đến D
    answer_match = re.search(r"Answer:\s*([A-D])", text)
    # Nếu tìm thấy
    if answer_match:
        # Trích xuất chữ cái đáp án (nhóm 1 trong regex) và lưu vào dictionary
        sections["answer_letter"] = answer_match.group(1)

    # Split the text into content before the answer
    # Chia văn bản thành các phần, lấy phần sau "SAT READING COMPREHENSION TEST" và trước "Answer:"
    # sau đó loại bỏ khoảng trắng ở đầu và cuối
    content = text.split("SAT READING COMPREHENSION TEST")[-1].split("Answer:")[0].strip()
    # Chia phần nội dung thành các khối dựa trên hai dòng trống liên tiếp
    # Loại bỏ các khối rỗng
    blocks = [b.strip() for b in content.split("\n\n") if b.strip()]

    # Khởi tạo danh sách để lưu trữ các dòng của bài đọc và các dòng của câu hỏi/lựa chọn
    passage_lines = []
    question_and_choices_lines = []
    # Biến cờ để kiểm tra xem đang ở trong phần câu hỏi hay không
    in_question_section = False

    # Lặp qua từng khối đã tách
    for block in blocks:
        # Nếu khối bắt đầu bằng "Question"
        if block.startswith("Question"):
            # Đặt cờ là True, cho biết đã vào phần câu hỏi
            in_question_section = True
            # Thêm khối này vào danh sách các dòng của câu hỏi và lựa chọn
            question_and_choices_lines.append(block)
        # Nếu đang ở trong phần câu hỏi
        elif in_question_section:
             # Assuming choices start with A), B), C), D) and are part of the question block
            # Giả định rằng các lựa chọn cũng nằm trong cùng khối văn bản với câu hỏi
            # Thêm khối này vào danh sách các dòng của câu hỏi và lựa chọn
            question_and_choices_lines.append(block)
        # Nếu không bắt đầu bằng "Question" và không ở trong phần câu hỏi
        else:
            # Thêm khối này vào danh sách các dòng của bài đọc
            passage_lines.append(block)

    # Nối các dòng của bài đọc lại với nhau, cách nhau bằng hai dòng trống
    # và loại bỏ khoảng trắng ở đầu và cuối, sau đó lưu vào dictionary
    sections["passage"] = "\n\n".join(passage_lines).strip()

    # Nếu có các dòng câu hỏi và lựa chọn
    if question_and_choices_lines:
        # Nối các dòng của câu hỏi và lựa chọn thành một chuỗi lớn
        question_block_text = "\n".join(question_and_choices_lines)
        # Find the first line that looks like a question start
        # Tìm kiếm dòng bắt đầu của câu hỏi (ví dụ: "Question 1)...")
        question_start_match = re.search(r"Question \d+.*?\)", question_block_text, re.DOTALL)

        # Nếu tìm thấy dòng bắt đầu câu hỏi
        if question_start_match:
             # Extract the question text after the question number and ')'
            # Tách chuỗi dựa vào ký tự ')', lấy phần sau
            q_part = question_block_text.split(")", 1)
            # Lấy dòng đầu tiên của phần sau ')' làm nội dung câu hỏi và loại bỏ khoảng trắng
            sections["question"] = q_part[-1].split("\n")[0].strip()

            # Extract choices, assuming they start with A), B), C), D) on new lines
            # Lấy tất cả các dòng trong khối câu hỏi và lựa chọn, sau đó lọc ra các dòng
            # bắt đầu bằng "A)", "B)", "C)", "D)" để làm các lựa chọn
            choice_lines = [line.strip() for line in question_block_text.split("\n")[1:]
                            if line.strip().startswith(("A)", "B)", "C)", "D)"))]
            # Lưu danh sách các lựa chọn vào dictionary
            sections["choices"] = choice_lines

    # Trả về dictionary chứa các phần đã trích xuất
    return sections


def map_answer(text, letter):
    """
    Finds the full text of the answer choice based on the answer letter.

    Args:
        text (str): The input text block containing the SAT reading comprehension problem.
        letter (str): The answer letter (e.g., "A", "B", "C", "D").

    Returns:
        str: The full text of the answer choice, or the original letter if not found.
    """
    # Trích xuất các phần từ văn bản gốc sử dụng hàm extract_sections
    sections = extract_sections(text)
    # Lặp qua từng lựa chọn trong danh sách các lựa chọn đã trích xuất
    for choice in sections["choices"]:
        # Nếu lựa chọn hiện tại bắt đầu bằng chữ cái đáp án được cung cấp (ví dụ: "A)")
        if choice.startswith(f"{letter})"):
            # Trả về toàn bộ văn bản của lựa chọn đó
            return choice
    # Nếu không tìm thấy lựa chọn nào bắt đầu bằng chữ cái đáp án
    # Trả về chữ cái đáp án gốc
    return letter


Đoạn code bạn cung cấp là một hàm Python có tên `generate_prompt`. Hàm này được thiết kế để tạo ra một prompt (đầu vào) cho một mô hình ngôn ngữ lớn (LLM), cụ thể là mô hình Llama 3, dựa trên dữ liệu của một bài đọc hiểu SAT.

Dưới đây là giải thích chi tiết về ý nghĩa của đoạn code:

**Ý nghĩa của đoạn code:**

1.  **Mục đích chính:** Hàm `generate_prompt` nhận vào văn bản gốc của một bài đọc hiểu SAT (`text`) và chữ cái đáp án đúng (`answer_letter`). Nhiệm vụ của nó là định dạng lại dữ liệu này thành một cấu trúc prompt phù hợp cho việc huấn luyện hoặc đánh giá mô hình Llama 3 theo định dạng hội thoại (conversational format).
2.  **Trích xuất thông tin:** Hàm đầu tiên gọi hàm `extract_sections(text)` (đã được định nghĩa ở phần trước trong file gốc của bạn). Hàm `extract_sections` có nhiệm vụ phân tích văn bản gốc để tách riêng phần bài đọc (`passage`), phần câu hỏi (`question`), danh sách các lựa chọn đáp án (`choices`) và chữ cái đáp án đúng (`answer_letter`) từ văn bản thô. Kết quả được lưu trong dictionary `sections`.
3.  **Định dạng các lựa chọn:** Các lựa chọn đáp án (`sections['choices']`) là một danh sách các chuỗi. Đoạn code sử dụng `"\n".join(sections['choices'])` để nối các lựa chọn này lại thành một chuỗi duy nhất, với mỗi lựa chọn nằm trên một dòng mới. Chuỗi này được lưu vào biến `choices_text`.
4.  **Tạo cấu trúc prompt hội thoại:** Phần chính của hàm là trả về một danh sách các dictionary. Mỗi dictionary trong danh sách này đại diện cho một lượt trong cuộc hội thoại giữa "hệ thống", "người dùng" và "trợ lý", đây là định dạng prompt phổ biến cho các mô hình ngôn ngữ dạng Instruct như Llama 3.

    *   **Vai trò 'system':** Dictionary đầu tiên có `role: "system"`. Nội dung (`content`) của nó là `LLAMA3_SYSTEM_PROMPT`, chứa các chỉ dẫn chung cho mô hình (ví dụ: "You are a helpful AI assistant..."). Đây là cách để hướng dẫn mô hình hành xử theo một vai trò nhất định.
    *   **Vai trò 'user':** Dictionary thứ hai có `role: "user"`. Nội dung (`content`) của nó chứa yêu cầu cụ thể từ người dùng. Trong trường hợp này, yêu cầu là "Read the passage and answer the question." (Đọc đoạn văn và trả lời câu hỏi). Sau đó, nó trình bày bài đọc, câu hỏi và các lựa chọn dưới các tiêu đề rõ ràng (`### Passage`, `### Question`, `### Choices`) sử dụng f-string để chèn nội dung đã trích xuất từ `sections` và `choices_text`. Cuối cùng, nó yêu cầu mô hình phản hồi theo định dạng cụ thể: "Respond with ONLY the letter and full text of the correct answer." (Trả lời CHỈ bằng chữ cái và toàn bộ văn bản của đáp án đúng).
    *   **Vai trò 'assistant':** Dictionary thứ ba có `role: "assistant"`. Nội dung (`content`) của nó là đáp án mong muốn mà mô hình nên tạo ra nếu được huấn luyện hoặc đánh giá với prompt này. Nó sử dụng hàm `map_answer(text, answer_letter)` (cũng đã được định nghĩa ở phần trước) để lấy toàn bộ văn bản của lựa chọn đáp án đúng dựa trên chữ cái đáp án được cung cấp. Đây chính là "nhãn" hay "đáp án mục tiêu" cho việc huấn luyện hoặc đánh giá.

**Về việc liệu đây có phải là template không thể thay đổi:**

Không, đây **không phải là một template không thể thay đổi được**.

*   **Ngữ cảnh sử dụng:** Đoạn code này là một hàm Python. Bạn hoàn toàn có thể chỉnh sửa nội dung bên trong hàm này, thay đổi cấu trúc của danh sách các dictionary, thay đổi văn bản trong phần `content` của các vai trò, hoặc thậm chí thêm/bớt các vai trò nếu mô hình của bạn hỗ trợ.
*   **Định dạng prompt của Llama 3 Instruct:** Định dạng hội thoại với các vai trò `system`, `user`, `assistant` là định dạng khuyến nghị (và thường là bắt buộc) khi làm việc với các phiên bản "Instruct" (được tinh chỉnh để làm theo hướng dẫn) của các mô hình như Llama 3. Mô hình được huấn luyện để hiểu và phản hồi theo cấu trúc này. Do đó, việc sử dụng cấu trúc này là cần thiết để mô hình hoạt động đúng cách. Tuy nhiên, nội dung cụ thể bên trong các phần `content` là do bạn định nghĩa.
*   **Mục đích fine-tuning:** Nếu bạn đang sử dụng đoạn code này để chuẩn bị dữ liệu cho việc fine-tuning (tinh chỉnh) mô hình Llama 3, cấu trúc prompt này đóng vai trò là cặp (đầu vào, đầu ra mục tiêu). Bạn đang "dạy" mô hình rằng khi nhận prompt dạng `system + user` như vậy, nó nên tạo ra nội dung dạng `assistant` tương ứng. Bạn có thể thay đổi cách bạn trình bày bài đọc, câu hỏi, lựa chọn trong phần `user` hoặc thay đổi định dạng của đáp án mục tiêu trong phần `assistant`, miễn là bạn nhất quán với định dạng bạn muốn mô hình học.

**Tóm lại:**

Đoạn code `generate_prompt` là một hàm tùy chỉnh được viết bằng Python để định dạng dữ liệu bài đọc hiểu SAT thành cấu trúc prompt hội thoại phù hợp cho mô hình Llama 3 Instruct. Mặc dù nó tuân theo định dạng hội thoại chuẩn của Llama 3 Instruct (với các vai trò `system`, `user`, `assistant`), nội dung chi tiết bên trong các phần `content` hoàn toàn có thể được bạn chỉnh sửa để phù hợp với yêu cầu cụ thể của bạn (ví dụ: thay đổi lời dẫn, cách trình bày câu hỏi/lựa chọn, định dạng đầu ra mong muốn). Nó không phải là một template cứng nhắc không thể thay đổi.

In [10]:
# Định nghĩa prompt hệ thống cho mô hình Llama 3
LLAMA3_SYSTEM_PROMPT = """You are a helpful AI assistant developed by Meta. Respond
safely and accurately."""

def generate_prompt(text, answer_letter): # Hàm tạo prompt cho mô hình từ văn bản bài đọc và chữ cái đáp án
    sections = extract_sections(text) # Trích xuất các phần (bài đọc, câu hỏi, lựa chọn) từ văn bản gốc
    choices_text = "\n".join(sections['choices']) # Nối các lựa chọn thành một chuỗi, mỗi lựa chọn trên một dòng mới

    return [ # Trả về danh sách các dict đại diện cho các phần của prompt theo định dạng hội thoại
        {
            "role": "system", # Vai trò: hệ thống (chỉ dẫn chung cho mô hình)
            "content": LLAMA3_SYSTEM_PROMPT # Nội dung chỉ dẫn hệ thống
        },
        {
            "role": "user", # Vai trò: người dùng (câu hỏi hoặc yêu cầu từ người dùng)
            "content": f"""Read the passage and answer the question. # Nội dung yêu cầu người dùng

### Passage: # Tiêu đề phần bài đọc
{sections["passage"]} # Nội dung bài đọc đã trích xuất

### Question: # Tiêu đề phần câu hỏi
{sections["question"]} # Nội dung câu hỏi đã trích xuất

### Choices: # Tiêu đề phần các lựa chọn
{choices_text} # Nội dung các lựa chọn đã nối chuỗi

Respond with ONLY the letter and full text of the correct answer.""" # Yêu cầu mô hình chỉ trả lời bằng chữ cái và toàn bộ văn bản của đáp án đúng
        },
        {
            "role": "assistant", # Vai trò: trợ lý (đáp án hoặc phản hồi mong muốn từ mô hình)
            "content": map_answer(text, answer_letter) # Nội dung đáp án đúng (văn bản đầy đủ) dựa vào chữ cái đáp án và văn bản gốc
        }
    ]




** `generate_and_tokenize_prompt(user_input, answer)`**

*   **Mục đích:** Hàm này là hàm chính để xử lý một mẫu dữ liệu (một bài đọc hiểu SAT đầy đủ với đáp án). Nó nhận vào văn bản thô của bài toán (`user_input`) và chữ cái đáp án đúng (`answer`), sau đó tạo ra prompt dạng hội thoại, định dạng nó thành một chuỗi và cuối cùng chuyển chuỗi đó thành các token số (numeric tokens) mà mô hình có thể xử lý. Nó cũng tạo ra nhãn (labels) cho việc huấn luyện.
*   **Cách hoạt động:**
    *   Nó sử dụng khối `try...except` để xử lý lỗi có thể xảy ra trong quá trình xử lý.
    *   Bên trong `try`:
        *   Nó gọi `full_prompt = generate_prompt(user_input, answer)` để tạo ra prompt dạng danh sách các dictionary như đã giải thích ở trên.
        *   Nó sử dụng `tokenizer.apply_chat_template(full_prompt, tokenize=False, add_generation_prompt=False)`: Đây là bước quan trọng. Tokenizer (đã được tải trước đó, ví dụ: `AutoTokenizer.from_pretrained(MODEL_NAME)`) có phương thức `apply_chat_template`. Phương thức này lấy danh sách các dictionary prompt (`full_prompt`) và chuyển nó thành một chuỗi văn bản duy nhất theo định dạng mà mô hình Llama 3 mong đợi (ví dụ: thêm các token đặc biệt như `<s>`, `[INST]`, `<<SYS>>`, `[/INST]`, `<|eot_id|>`).
            *   `tokenize=False`: Chỉ định rằng *chỉ* định dạng chuỗi, chưa chuyển thành token số ở bước này. Việc tokenize sẽ được thực hiện ở bước tiếp theo.
            *   `add_generation_prompt=False`: Chỉ định rằng không thêm các token báo hiệu bắt đầu cho quá trình sinh văn bản (ví dụ: `[INST]`). Điều này phù hợp khi bạn chuẩn bị dữ liệu cho việc huấn luyện (fine-tuning).
            *   Kết quả là một chuỗi `prompt_str` đã được định dạng theo template chat của Llama 3.
        *   Nó sử dụng `tokenized = tokenizer(...)`: Bước này thực sự chuyển đổi chuỗi `prompt_str` thành các token số (input IDs) và các tensor khác cần thiết cho mô hình.
            *   `prompt_str`: Chuỗi văn bản đã được định dạng.
            *   `padding="max_length"`: Thêm token pad vào cuối chuỗi nếu độ dài nhỏ hơn `max_length`. Điều này đảm bảo tất cả các chuỗi đầu vào có cùng kích thước, cần thiết cho việc xử lý theo batch.
            *   `truncation=True`: Cắt bớt chuỗi nếu độ dài vượt quá `max_length`.
            *   `max_length=1024`: Chiều dài tối đa của chuỗi tokenized.
            *   `return_tensors="pt"`: Trả về kết quả dưới dạng PyTorch tensors.
        *   `input_ids = tokenized["input_ids"][0]`: Lấy tensor chứa các token ID từ kết quả tokenizer. `[0]` ở đây giả định rằng bạn chỉ xử lý một mẫu một lúc hoặc lấy mẫu đầu tiên từ batch.
        *   `labels = input_ids.clone()`: Trong tác vụ Causal Language Modeling (như fine-tuning Llama 3), mô hình được huấn luyện để dự đoán token tiếp theo. Do đó, nhãn (label) cho một token đầu vào chính là token tiếp theo trong chuỗi. Cách phổ biến là sử dụng `input_ids` làm `labels`, và sau đó mask (ẩn đi) phần prompt ban đầu để mô hình chỉ tính toán loss trên phần phản hồi của assistant. Đoạn code này tạo ra nhãn ban đầu bằng cách sao chép `input_ids`. Việc masking sẽ thường được thực hiện sau này trong quá trình huấn luyện.
        *   Hàm trả về một dictionary chứa `"input_ids"`, `"attention_mask"` (tensor chỉ định token nào nên được chú ý, thường 1 cho token thực và 0 cho pad token) và `"labels"`.
    *   Trong khối `except`: Nếu bất kỳ lỗi nào xảy ra (ví dụ: `extract_sections` hoặc `map_answer` gặp vấn đề với định dạng văn bản), lỗi sẽ được in ra và hàm trả về `None`, báo hiệu rằng mẫu này không thể xử lý được.

**Tóm lại:**

Chuỗi các hàm này hoạt động cùng nhau để lấy dữ liệu văn bản thô của bài đọc hiểu SAT, phân tích nó, định dạng lại thành một cấu trúc prompt hội thoại chuẩn của Llama 3, và sau đó chuyển cấu trúc prompt đã định dạng đó thành các tensor số (`input_ids`, `attention_mask`, `labels`) sẵn sàng để đưa vào mô hình ngôn ngữ lớn để huấn luyện hoặc đánh giá.

In [11]:
def generate_and_tokenize_prompt(user_input, answer):
  """
    Generates a full prompt using a chat template and tokenizes it.

    Args:
        user_input (str): The raw text of the SAT reading comprehension problem.
        answer (str): The correct answer letter for the problem.

    Returns:
        dict or None: A dictionary containing input_ids, attention_mask, and labels
                      if successful, otherwise None.
  """
  try:
    # generate_prompt function is assumed to be defined elsewhere and returns a list of dicts
    full_prompt = generate_prompt(user_input, answer)

    # Apply the chat template to format the prompt into a single string
    prompt_str = tokenizer.apply_chat_template(
        full_prompt,
        tokenize=False, # Do not tokenize here, we will tokenize later
        add_generation_prompt=False # Do not add the prompt for generation start
    )

    # Tokenize the formatted prompt string
    tokenized = tokenizer(
        prompt_str,
        padding="max_length", # Pad sequences to the max_length
        truncation=True, # Truncate sequences longer than max_length
        max_length=1024, # The maximum sequence length
        return_tensors="pt" # Return PyTorch tensors
    )

    # Extract the input IDs and create labels (for language modeling, labels are the same as input_ids)
    input_ids = tokenized["input_ids"][0]
    labels = input_ids.clone() # Create a copy of input_ids for labels

    # Return the dictionary containing input_ids, attention_mask, and labels
    return {
        "input_ids": input_ids,
        "attention_mask": tokenized["attention_mask"][0],
        "labels": labels
    }

  except Exception as e:
    # Catch any exceptions and print an error message
    print(f"Error processing sample: {e}")
    return None # Return None if an error occurs


In [12]:
import re
from sklearn.model_selection import train_test_split
from huggingface_hub import login
# Import the function explicitly to ensure it's recognized
from __main__ import generate_and_tokenize_prompt, map_answer, extract_sections

training_samples = []
for sample in tqdm(data["train"]):
  try:
    # Clean and process the text and answer
    # Use the imported functions
    processed_text = sample["text"].replace("SAT READING COMPREHENSION TEST", "").strip()
    processed_answer = map_answer(sample["text"], sample["answer"].strip())

    # Generate and tokenize the prompt
    tokenized_sample = generate_and_tokenize_prompt(processed_text, processed_answer)

    # Add the processed sample to the training list if tokenization was successful
    if tokenized_sample is not None:
      training_samples.append(tokenized_sample)
  except Exception as e:
    # Skip invalid samples and print an error message
    print(f"Skipping invalid sample: {e}")

# Filter out any None values that might have resulted from errors during processing
training_samples = [s for s in training_samples if s is not None]

# Add a check for empty training_samples before splitting
if not training_samples:
    print("Error: No valid training samples were generated. Cannot split the dataset.")
else:
    # Split the data into training and validation sets
    train_samples, val_samples = train_test_split(training_samples, test_size=0.1, random_state=42)

    # Convert the sample lists into Hugging Face Dataset objects
    train_dataset = Dataset.from_list(train_samples)
    eval_dataset = Dataset.from_list(val_samples)

100%|██████████| 298/298 [00:01<00:00, 155.02it/s]


In [14]:
from transformers import TrainerCallback, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch # Import torch if it's not already imported in the provided context
from peft import PeftModel # Import PeftModel to check the model type

class LogLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            print(f"Step {state.global_step} - Loss: {logs['loss']:.4f}")

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    save_total_limit=3,
    logging_steps=10,
    output_dir="llama3-8b-sat-reading",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

# Add a check to ensure the model is a PeftModel
if not isinstance(model, PeftModel):
    raise TypeError("The model is not a PEFT model. Please ensure PEFT is applied before training.")

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    data_collator=data_collator,
    callbacks=[LogLossCallback()]
)

model.config.use_cache = False
model.enable_input_require_grads()
# The torch.compile call might interfere with PEFT and quantization.
# It's often recommended to train without torch.compile first,
# especially when debugging. I will comment it out for now.
# model = torch.compile(model)


trainer.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


Step 10 - Loss: 1.6991
Step 20 - Loss: 1.9022
Step 30 - Loss: 1.6926
Step 40 - Loss: 1.5011
Step 50 - Loss: 1.6307


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.96 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.38 GiB is free. Process 3762 has 13.36 GiB memory in use. Of the allocated memory 10.57 GiB is allocated by PyTorch, and 2.65 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
model.save_pretrained("trained-model")
PEFT_MODEL = "your_huggingface_user_name/instructionTuning-llama-3-1-8B-SAT-readingsolver"
model.push_to_hub(PEFT_MODEL, use_auth_token=True)


In [ ]:

PEFT_MODEL = "your_huggingface_user_name/instructionTuning-llama-3-1-8B-SAT-readingsolver"

config = PeftConfig.from_pretrained(PEFT_MODEL)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(model, PEFT_MODEL)


In [ ]:
from transformers import GenerationConfig # Import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=64,
    temperature=0.0,
    top_p=1.0,
    do_sample=False,
    repetition_penalty=1.0,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

def format_test_prompt(text):
    """
    Formats the input text for testing by extracting sections and creating a user prompt.

    Args:
        text (str): The raw text of the SAT reading comprehension problem.

    Returns:
        list: A list of dictionaries representing the formatted chat prompt.
    """
    sections = extract_sections(text)
    choices_text = "\n".join(sections['choices'])

    return [
        {
            "role": "system",
            "content": LLAMA3_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""Read the passage and answer the question.

### Passage:
{sections["passage"]}

### Question:
{sections["question"]}

### Choices:
{choices_text}

Respond with ONLY the letter and full text of the correct answer."""
        }
    ]


def extract_answer(output_text):
    """
    Extracts the predicted answer choice from the model's output text.

    Args:
        output_text (str): The raw text output from the model.

    Returns:
        str: The extracted answer choice text, or the full output if not found.
    """
    # Assuming the model output starts with the letter and text, e.g., "A) Some answer text"
    # Find the first occurrence of a choice letter followed by ')'.
    match = re.search(r"[A-D]\)", output_text)
    if match:
        # Return the text from the start of the match onwards.
        return output_text[match.start():].strip()
    else:
        # If no standard choice format is found, return the whole output as the potential answer.
        return output_text.strip()

def extract_choice_letter(answer_text):
  """
  Extracts the answer choice letter (A, B, C, D) from the answer text.

  Args:
    answer_text (str): The text of the answer (e.g., "A) choice text").

  Returns:
    str: The extracted letter (e.g., "A"), or None if not found.
  """
  match = re.match(r"([A-D])\)", answer_text)
  if match:
    return match.group(1)
  return None


def predict(text):
    """
    Generates a prediction for a given SAT reading comprehension problem.

    Args:
        text (str): The raw text of the SAT reading comprehension problem.

    Returns:
        str: The predicted answer text.
    """
    messages = format_test_prompt(text)

    prompt_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            generation_config=generation_config
        )

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # The output contains the input prompt as well, we need to find where the model's generation starts
    # A simple way is to find the end of the assistant's turn prompt
    # Note: This is a simplification. A more robust method might involve finding the exact prompt length.
    assistant_start_marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    if assistant_start_marker in output_text:
        output_text = output_text.split(assistant_start_marker, 1)[1]

    return extract_answer(output_text)

for i in range(10):
    print("="*100)
    sample = data["test"][i]
    input_text = sample["text"]
    true_answer = sample["answer"].strip()

    predicted_answer = predict(input_text)

    true_answer_full = map_answer(input_text, true_answer)

    pred_choice = extract_choice_letter(predicted_answer)
    true_choice = extract_choice_letter(true_answer_full)

    print(f"### Sample {i+1}")
    print(f"[Question]\n{extract_sections(input_text)['question']}")
    print(f"[Choices]\n{'\n'.join(extract_sections(input_text)['choices'])}")
    print(f"\n[Model Prediction]\n{predicted_answer}")
    print(f"\n[Ground Truth]\n{true_answer_full}")
    print(f"""\nResult: {"CORRECT" if pred_choice == true_choice else "INCORRECT"}""")
    print("="*100 + "\n")


## Test các hàm trên

In [ ]:
#data['train'][0]
print(data['train'][0])

{'text': 'SAT READING COMPREHENSION TEST\n\nThis passage is adapted from George Eliot, Silas Marner.\nOriginally published in 1861. Silas was a weaver and a\nnotorious miser, but then the gold he had hoarded was\nstolen. Shortly after, Silas adopted a young child, Eppie, the\ndaughter of an impoverished woman who had died\nsuddenly.\n\n    Unlike the gold which needed nothing, and must\nbe worshipped in close-locked solitude—which was\nhidden away from the daylight, was deaf to the song\nof birds, and started to no human tones—Eppie was a\ncreature of endless claims and ever-growing desires,\nseeking and loving sunshine, and living sounds, and\nliving movements; making trial of everything, with\ntrust in new joy, and stirring the human kindness in\nall eyes that looked on her. The gold had kept his\nthoughts in an ever-repeated circle, leading to\nnothing beyond itself; but Eppie was an object\ncompacted of changes and hopes that forced his\nthoughts onward, and carried them far away f

In [ ]:
import re
from sklearn.model_selection import train_test_split
from huggingface_hub import login
# Import the function explicitly to ensure it's recognized
from __main__ import generate_and_tokenize_prompt, map_answer, extract_sections
from tqdm import tqdm
from datasets import Dataset # Import Dataset



# The variable 'text' is a dictionary, we need to access its 'text' key
raw_text = data['train'][0]['text']

answer_match = re.search(r"Answer:\s*([A-D])", raw_text)
    # Nếu tìm thấy
    if answer_match:
        # Trích xuất chữ cái đáp án (nhóm 1 trong regex) và lưu vào dictionary
        sections["answer_letter"] = answer_match.group(1)



sections = {
    "passage": "",       # Khởi tạo phần bài đọc là chuỗi rỗng
    "question": "",      # Khởi tạo phần câu hỏi là chuỗi rỗng
    "choices": [],       # Khởi tạo phần các lựa chọn là danh sách rỗng
    "answer_letter": ""  # Khởi tạo phần chữ cái đáp án là chuỗi rỗng
}


# Use raw_text instead of text
answer_match = re.search(r"Answer:\s*([A-D])", raw_text)
print(answer_match)
if answer_match:
    # Trích xuất chữ cái đáp án (nhóm 1 trong regex) và lưu vào dictionary
    sections["answer_letter"] = answer_match.group(1)

# Split the text into content before the answer
# Chia văn bản thành các phần, lấy phần sau "SAT READING COMPREHENSION TEST" và trước "Answer:"
# sau đó loại bỏ khoảng trắng ở đầu và cuối
# Use raw_text instead of text
content = raw_text.split("SAT READING COMPREHENSION TEST")[-1].split("Answer:")[0].strip()

# Chia phần nội dung thành các khối dựa trên hai dòng trống liên tiếp
# Loại bỏ các khối rỗng
blocks = [b.strip() for b in content.split("\n\n") if b.strip()]

# Khởi tạo danh sách để lưu trữ các dòng của bài đọc và các dòng của câu hỏi/lựa chọn
passage_lines = []
question_and_choices_lines = []
# Biến cờ để kiểm tra xem đang ở trong phần câu hỏi hay không
in_question_section = False

# Lặp qua từng khối đã tách
for block in blocks:
    # Nếu khối bắt đầu bằng "Question"
    if block.startswith("Question"):
        # Đặt cờ là True, cho biết đã vào phần câu hỏi
        in_question_section = True
        # Thêm khối này vào danh sách các dòng của câu hỏi và lựa chọn
        question_and_choices_lines.append(block)
    # Nếu đang ở trong phần câu hỏi
    elif in_question_section:
          # Assuming choices start with A), B), C), D) and are part of the question block
        # Giả định rằng các lựa chọn cũng nằm trong cùng khối văn bản với câu hỏi
        # Thêm khối này vào danh sách các dòng của câu hỏi và lựa chọn
        question_and_choices_lines.append(block)
    # Nếu không bắt đầu bằng "Question" và không ở trong phần câu hỏi
    else:
        # Thêm khối này vào danh sách các dòng của bài đọc
        passage_lines.append(block)

# Nối các dòng của bài đọc lại với nhau, cách nhau bằng hai dòng trống
# và loại bỏ khoảng trắng ở đầu và cuối, sau đó lưu vào dictionary
sections["passage"] = "\n\n".join(passage_lines).strip()

# Nếu có các dòng câu hỏi và lựa chọn
if question_and_choices_lines:
    # Nối các dòng của câu hỏi và lựa chọn thành một chuỗi lớn
    question_block_text = "\n".join(question_and_choices_lines)
    # Find the first line that looks like a question start
    # Tìm kiếm dòng bắt đầu của câu hỏi (ví dụ: "Question 1)...")
    question_start_match = re.search(r"Question \d+.*?\)", question_block_text, re.DOTALL)

    # Nếu tìm thấy dòng bắt đầu câu hỏi
    if question_start_match:
          # Extract the question text after the question number and ')'
        # Tách chuỗi dựa vào ký tự ')', lấy phần sau
        q_part = question_block_text.split(")", 1)
        # Lấy dòng đầu tiên của phần sau ')' làm nội dung câu hỏi và loại bỏ khoảng trắng
        sections["question"] = q_part[-1].split("\n")[0].strip()

        # Extract choices, assuming they start with A), B), C), D) on new lines
        # Lấy tất cả các dòng trong khối câu hỏi và lựa chọn, sau đó lọc ra các dòng
        # bắt đầu bằng "A)", "B)", "C)", "D)" để làm các lựa chọn
        choice_lines = [line.strip() for line in question_block_text.split("\n")[1:]
                        if line.strip().startswith(("A)", "B)", "C)", "D)"))]
        # Lưu danh sách các lựa chọn vào dictionary
        sections["choices"] = choice_lines

IndentationError: unexpected indent (<ipython-input-13-6ad78a5de13a>, line 16)

In [ ]:
def map_answer(text, letter):
    """
    Finds the full text of the answer choice based on the answer letter.

    Args:
        text (str): The input text block containing the SAT reading comprehension problem.
        letter (str): The answer letter (e.g., "A", "B", "C", "D").

    Returns:
        str: The full text of the answer choice, or the original letter if not found.
    """
    # Trích xuất các phần từ văn bản gốc sử dụng hàm extract_sections
    sections = extract_sections(text)
    # Lặp qua từng lựa chọn trong danh sách các lựa chọn đã trích xuất
    for choice in sections["choices"]:
        # Nếu lựa chọn hiện tại bắt đầu bằng chữ cái đáp án được cung cấp (ví dụ: "A)")
        if choice.startswith(f"{letter})"):
            # Trả về toàn bộ văn bản của lựa chọn đó
            return choice
    # Nếu không tìm thấy lựa chọn nào bắt đầu bằng chữ cái đáp án
    # Trả về chữ cái đáp án gốc
    return letter

In [ ]:
content = raw_text.split("SAT READING COMPREHENSION TEST")[-1].split("Answer:")[0].strip()
#print(raw_text.split("SAT READING COMPREHENSION TEST"))
print(raw_text.split("SAT READING COMPREHENSION TEST")[-1])
print('############################################')
print(raw_text.split("SAT READING COMPREHENSION TEST")[-1].split("Answer:")[0])
print('############################################')
print(raw_text.split("SAT READING COMPREHENSION TEST")[-1].split("Answer:")[0].strip())
print('############################################')



This passage is adapted from George Eliot, Silas Marner.
Originally published in 1861. Silas was a weaver and a
notorious miser, but then the gold he had hoarded was
stolen. Shortly after, Silas adopted a young child, Eppie, the
daughter of an impoverished woman who had died
suddenly.

    Unlike the gold which needed nothing, and must
be worshipped in close-locked solitude—which was
hidden away from the daylight, was deaf to the song
of birds, and started to no human tones—Eppie was a
creature of endless claims and ever-growing desires,
seeking and loving sunshine, and living sounds, and
living movements; making trial of everything, with
trust in new joy, and stirring the human kindness in
all eyes that looked on her. The gold had kept his
thoughts in an ever-repeated circle, leading to
nothing beyond itself; but Eppie was an object
compacted of changes and hopes that forced his
thoughts onward, and carried them far away from
their old eager pacing towards the same blank
limit—carri

In [ ]:
content = raw_text.split("SAT READING COMPREHENSION TEST")[-1].split("Answer:")[0].strip()
blocks = [b.strip() for b in content.split("\n\n") if b.strip()]
print(content)
blocks

This passage is adapted from George Eliot, Silas Marner.
Originally published in 1861. Silas was a weaver and a
notorious miser, but then the gold he had hoarded was
stolen. Shortly after, Silas adopted a young child, Eppie, the
daughter of an impoverished woman who had died
suddenly.

    Unlike the gold which needed nothing, and must
be worshipped in close-locked solitude—which was
hidden away from the daylight, was deaf to the song
of birds, and started to no human tones—Eppie was a
creature of endless claims and ever-growing desires,
seeking and loving sunshine, and living sounds, and
living movements; making trial of everything, with
trust in new joy, and stirring the human kindness in
all eyes that looked on her. The gold had kept his
thoughts in an ever-repeated circle, leading to
nothing beyond itself; but Eppie was an object
compacted of changes and hopes that forced his
thoughts onward, and carried them far away from
their old eager pacing towards the same blank
limit—carried

['This passage is adapted from George Eliot, Silas Marner.\nOriginally published in 1861. Silas was a weaver and a\nnotorious miser, but then the gold he had hoarded was\nstolen. Shortly after, Silas adopted a young child, Eppie, the\ndaughter of an impoverished woman who had died\nsuddenly.',
 'Unlike the gold which needed nothing, and must\nbe worshipped in close-locked solitude—which was\nhidden away from the daylight, was deaf to the song\nof birds, and started to no human tones—Eppie was a\ncreature of endless claims and ever-growing desires,\nseeking and loving sunshine, and living sounds, and\nliving movements; making trial of everything, with\ntrust in new joy, and stirring the human kindness in\nall eyes that looked on her. The gold had kept his\nthoughts in an ever-repeated circle, leading to\nnothing beyond itself; but Eppie was an object\ncompacted of changes and hopes that forced his\nthoughts onward, and carried them far away from\ntheir old eager pacing towards the same 

In [ ]:
# The variable 'text' is a dictionary, we need to access its 'text' key
import re
raw_text = data['train'][0]['text']

answer_match = re.search(r"Answer:\s*([A-D])", raw_text)
print(answer_match)
# Nếu tìm thấy
if answer_match:
    # Trích xuất chữ cái đáp án (nhóm 1 trong regex) và lưu vào dictionary
    sections["answer_letter"] = answer_match.group(1)

None
